In [12]:

%pip install -q langgraph


In [13]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Shared state
class MeetingState(TypedDict):
    topic: str
    date: str
    time: str
    result: str

# Single Agent
def meeting_agent(state: MeetingState):
    print("AI Agent is working...")

    topic = state["topic"]
    date = state["date"]
    time = state["time"]

    result = (
        f"Meeting Topic: {topic}\n"
        f"Meeting Date: {date}\n"
        f"Meeting Time: {time}\n"
        "Status: Meeting plan prepared successfully."
    )

    return {"result": result}

# Build the LangGraph workflow
builder = StateGraph(MeetingState)

builder.add_node("meeting_agent", meeting_agent)

builder.add_edge(START, "meeting_agent")
builder.add_edge("meeting_agent", END)

# Compile workflow
app = builder.compile()

print("Single Agent Workflow Created Successfully!")


Single Agent Workflow Created Successfully!


In [14]:

meeting_request = {
    "topic": "EMEA AI Training Team Meeting",
    "date": "2026-10-16",
    "time": "10:00 AM",
    "result": ""
}

response = app.invoke(meeting_request)

print("\n--- FINAL MEETING PLAN ---")
print(response["result"])


AI Agent is working...

--- FINAL MEETING PLAN ---
Meeting Topic: EMEA AI Training Team Meeting
Meeting Date: 2026-10-16
Meeting Time: 10:00 AM
Status: Meeting plan prepared successfully.


In [15]:

from datetime import datetime
from langchain_core.tools import tool

@tool
def validate_meeting_time(date: str, time: str) -> str:
    """Check whether a meeting date and time are valid."""

    try:
        meeting_date = datetime.strptime(date, "%Y-%m-%d").date()
        meeting_time = datetime.strptime(time, "%I:%M %p").time()

        if meeting_date < datetime.today().date():
            return "Invalid: Meeting date is in the past."

        if not (9 <= meeting_time.hour < 18):
            return "Warning: Meeting is outside working hours (9 AM–6 PM)."

        return "Valid: Meeting date and working hours are acceptable."

    except ValueError:
        return "Invalid: Please check the date or time format."


# Test the tool
result = validate_meeting_time.invoke({
    "date": "2026-10-16",
    "time": "10:00 AM"
})

print(result)


Valid: Meeting date and working hours are acceptable.


In [16]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Shared information between workflow nodes
class WorkflowState(TypedDict):
    topic: str
    date: str
    time: str
    validation: str
    final_response: str

# Node 1: Agent decides to validate the meeting
def agent_node(state: WorkflowState):
    print("Agent: Processing meeting request...")
    return {}

# Node 2: Execute the validation tool
def tool_node(state: WorkflowState):
    print("Tool: Validating meeting date and time...")

    result = validate_meeting_time.invoke({
        "date": state["date"],
        "time": state["time"]
    })

    return {"validation": result}

# Node 3: Agent prepares final response
def response_node(state: WorkflowState):
    print("Agent: Preparing final response...")

    response = f"""
MEETING PLAN
Topic: {state['topic']}
Date: {state['date']}
Time: {state['time']}
Validation: {state['validation']}
"""
    return {"final_response": response}

# Create the orchestrated workflow
workflow = StateGraph(WorkflowState)

workflow.add_node("agent", agent_node)
workflow.add_node("validation_tool", tool_node)
workflow.add_node("final_response", response_node)

workflow.add_edge(START, "agent")
workflow.add_edge("agent", "validation_tool")
workflow.add_edge("validation_tool", "final_response")
workflow.add_edge("final_response", END)

meeting_app = workflow.compile()

# Run the workflow
output = meeting_app.invoke({
    "topic": "EMEA AI Training Team Meeting",
    "date": "2026-10-16",
    "time": "10:00 AM",
    "validation": "",
    "final_response": ""
})

print(output["final_response"])


Agent: Processing meeting request...
Tool: Validating meeting date and time...
Agent: Preparing final response...

MEETING PLAN
Topic: EMEA AI Training Team Meeting
Date: 2026-10-16
Time: 10:00 AM
Validation: Valid: Meeting date and working hours are acceptable.



In [17]:

%pip install -q gradio


In [18]:

import gradio as gr

def create_meeting(topic, meeting_date, meeting_time, attendees):
    request = {
        "topic": topic,
        "date": meeting_date,
        "time": meeting_time,
        "validation": "",
        "final_response": ""
    }

    result = meeting_app.invoke(request)

    return (
        result["final_response"]
        + f"\nAttendees: {attendees}"
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Meeting Assistant")
    gr.Markdown("Powered by Python and LangGraph")

    topic = gr.Textbox(
        label="Meeting Topic",
        value="EMEA AI Training Team Meeting"
    )
    meeting_date = gr.Textbox(
        label="Date (YYYY-MM-DD)",
        value="2026-10-16"
    )
    meeting_time = gr.Textbox(
        label="Time (e.g. 10:00 AM)",
        value="10:00 AM"
    )
    attendees = gr.Textbox(
        label="Attendees",
        value="AI Training Team"
    )

    submit = gr.Button("Generate Meeting Plan")
    output = gr.Textbox(label="Meeting Plan", lines=8)

    submit.click(
        fn=create_meeting,
        inputs=[topic, meeting_date, meeting_time, attendees],
        outputs=output
    )

demo.launch(share=True)


/tmp/ipykernel_695/1099212175.py:19: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5101f3b9c5f3802190.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
